In [75]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# **Mlflow and dagshub initialisation**

In [65]:
!pip install -q dagshub mlflow xgboost

In [66]:
import dagshub
import mlflow

dagshub.init(repo_owner="luchit22", repo_name="ML-fraud-detection",mlflow=True)
mlflow.set_experiment("xgboost")

Initialized MLflow to track repo "luchit22/ML-fraud-detection"

Repository luchit22/ML-fraud-detection initialized!

2026/05/02 17:35:15 INFO mlflow.tracking.fluent: Experiment with name 'xgboost' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/7ce59dc35ae2449f9c775bcc51158ae8', creation_time=1777743315373, experiment_id='3', last_update_time=1777743315373, lifecycle_stage='active', name='xgboost', tags={}, trace_location=None, workspace='default'>

In [67]:
import mlflow.xgboost

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

# **data cleaning**

In [76]:
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

train_transaction.columns = train_transaction.columns.str.replace("-", "_")
train_identity.columns = train_identity.columns.str.replace("-", "_")

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)

(590540, 434)


In [78]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), object(31)
memory usage: 1.9+ GB


In [93]:
y = train["isFraud"]
X = train.drop(columns=["isFraud"])

In [94]:
missing_ratio = X.isnull().mean()

cols_to_drop = missing_ratio[missing_ratio > 0.90].index.tolist()

X = X.drop(columns=cols_to_drop)

print("Dropped columns:", len(cols_to_drop))
print("Shape after dropping:", X.shape)

Dropped columns: 12
Shape after dropping: (590540, 421)


# **feature engineering**

In [95]:
def add_features(df):
    df = df.copy()

    if "TransactionAmt" in df.columns:
        df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])

    if "TransactionDT" in df.columns:
        df["TransactionDT_days"] = df["TransactionDT"] / (3600 * 24)
        df["TransactionDT_hours"] = df["TransactionDT"] / 3600
        df["TransactionDT_dayofweek"] = (df["TransactionDT_days"] % 7).astype(int)

    df["num_missing"] = df.isnull().sum(axis=1)

    return df

In [96]:
X = add_features(X)

print("Shape after FE:", X.shape)


Shape after FE: (590540, 426)


In [97]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols:
    freq_map = X[col].value_counts(dropna=False).to_dict()
    X[col + "_freq"] = X[col].map(freq_map)

X = X.drop(columns=categorical_cols)

print("Shape after frequency encoding:", X.shape)

Shape after frequency encoding: (590540, 426)


In [98]:
X = X.replace([np.inf, -np.inf], np.nan)

# **baseline train**

In [99]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

Train: (472432, 426)
Valid: (118108, 426)


In [100]:
xgb_baseline = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [101]:
xgb_baseline.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=50
)

[0]	validation_0-auc:0.78755
[50]	validation_0-auc:0.89802
[100]	validation_0-auc:0.91158
[150]	validation_0-auc:0.91974
[200]	validation_0-auc:0.92463
[250]	validation_0-auc:0.92840
[300]	validation_0-auc:0.93257
[350]	validation_0-auc:0.93656
[400]	validation_0-auc:0.93947
[450]	validation_0-auc:0.94229
[499]	validation_0-auc:0.94418


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [102]:
train_proba = xgb_baseline.predict_proba(X_train)[:, 1]
valid_proba = xgb_baseline.predict_proba(X_valid)[:, 1]

train_auc = roc_auc_score(y_train, train_proba)
valid_auc = roc_auc_score(y_valid, valid_proba)
auc_gap = train_auc - valid_auc

print("Train ROC-AUC:", train_auc)
print("Validation ROC-AUC:", valid_auc)
print("AUC gap:", auc_gap)

Train ROC-AUC: 0.9598016137626164
Validation ROC-AUC: 0.9441770242316417
AUC gap: 0.015624589530974675


In [103]:
with mlflow.start_run(run_name="XGBoost_Baseline"):
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("tree_method", "hist")
    mlflow.log_param("encoding", "frequency_encoding")
    mlflow.log_param("dropped_missing_threshold", 0.90)
    mlflow.log_param("num_features", X_train.shape[1])

    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("valid_auc", valid_auc)
    mlflow.log_metric("auc_gap", auc_gap)

    mlflow.xgboost.log_model(xgb_baseline, artifact_path="xgboost_baseline")

2026/05/02 18:01:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost_Baseline at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3/runs/07029c3ab6364152b0330fc9a254475d
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3


# **feature selection**

In [104]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_baseline.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(30)

,feature,importance
295,V258,0.242885
238,V201,0.064477
107,V70,0.025084
16,C7,0.018389
128,V91,0.016170
193,V156,0.014360
331,V294,0.013203
354,V317,0.013082
332,V295,0.012834
17,C8,0.011001


In [105]:
top_features = importance_df.head(200)["feature"].tolist()

X_train_top = X_train[top_features]
X_valid_top = X_valid[top_features]

print("Selected features:", len(top_features))

Selected features: 200


In [106]:
corr_matrix = X_train_top.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    col for col in upper.columns
    if any(upper[col] > 0.95)
]

X_train_final = X_train_top.drop(columns=high_corr_features)
X_valid_final = X_valid_top.drop(columns=high_corr_features)

print("Removed correlated features:", len(high_corr_features))
print("Final feature count:", X_train_final.shape[1])

Removed correlated features: 56
Final feature count: 144


# **train**

In [107]:
xgb_configs = [
    {
        "name": "XGB_depth4_lr005",
        "n_estimators": 700,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 3,
        "gamma": 0,
        "reg_lambda": 1
    },
    {
        "name": "XGB_depth6_lr003",
        "n_estimators": 900,
        "max_depth": 6,
        "learning_rate": 0.03,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "min_child_weight": 3,
        "gamma": 0,
        "reg_lambda": 1
    },
    {
        "name": "XGB_depth8_lr002",
        "n_estimators": 1000,
        "max_depth": 8,
        "learning_rate": 0.02,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "gamma": 0.1,
        "reg_lambda": 2
    }
]

In [108]:
xgb_results = []
best_xgb = None
best_valid_auc = 0
best_config = None

for config in xgb_configs:
    with mlflow.start_run(run_name=config["name"]):
        model = XGBClassifier(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            learning_rate=config["learning_rate"],
            subsample=config["subsample"],
            colsample_bytree=config["colsample_bytree"],
            min_child_weight=config["min_child_weight"],
            gamma=config["gamma"],
            reg_lambda=config["reg_lambda"],
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train_final,
            y_train,
            eval_set=[(X_valid_final, y_valid)],
            verbose=100
        )

        train_proba = model.predict_proba(X_train_final)[:, 1]
        valid_proba = model.predict_proba(X_valid_final)[:, 1]

        train_auc = roc_auc_score(y_train, train_proba)
        valid_auc = roc_auc_score(y_valid, valid_proba)
        auc_gap = train_auc - valid_auc

        mlflow.log_param("model", "XGBoost")
        mlflow.log_params(config)
        mlflow.log_param("encoding", "frequency_encoding")
        mlflow.log_param("feature_selection", "xgboost_importance_top200 + correlation_filter")
        mlflow.log_param("correlation_threshold", 0.95)
        mlflow.log_param("num_final_features", X_train_final.shape[1])

        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("valid_auc", valid_auc)
        mlflow.log_metric("auc_gap", auc_gap)

        xgb_results.append({
            **config,
            "train_auc": train_auc,
            "valid_auc": valid_auc,
            "auc_gap": auc_gap
        })

        if valid_auc > best_valid_auc:
            best_valid_auc = valid_auc
            best_xgb = model
            best_config = config

xgb_results_df = pd.DataFrame(xgb_results).sort_values("valid_auc", ascending=False)
xgb_results_df

[0]	validation_0-auc:0.69736
[100]	validation_0-auc:0.88256
[200]	validation_0-auc:0.89573
[300]	validation_0-auc:0.90279
[400]	validation_0-auc:0.90811
[500]	validation_0-auc:0.91254
[600]	validation_0-auc:0.91571
[699]	validation_0-auc:0.91879
🏃 View run XGB_depth4_lr005 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3/runs/426c72698a794ba5a4158574150577c9
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3
[0]	validation_0-auc:0.78854
[100]	validation_0-auc:0.89189
[200]	validation_0-auc:0.90775
[300]	validation_0-auc:0.91627
[400]	validation_0-auc:0.92208
[500]	validation_0-auc:0.92661
[600]	validation_0-auc:0.93070
[700]	validation_0-auc:0.93447
[800]	validation_0-auc:0.93763
[899]	validation_0-auc:0.94051
🏃 View run XGB_depth6_lr003 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3/runs/175565788dca466ea352829becde2afb
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.m

,name,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,gamma,reg_lambda,train_auc,valid_auc,auc_gap
2,XGB_depth8_lr002,1000,8,0.02,0.80,0.80,5,0.1,2,0.966587,0.948160,0.018427
1,XGB_depth6_lr003,900,6,0.03,0.85,0.85,3,0.0,1,0.955420,0.940509,0.014911
0,XGB_depth4_lr005,700,4,0.05,0.90,0.90,3,0.0,1,0.929145,0.918794,0.010351


In [109]:
with mlflow.start_run(run_name="XGBoost_BEST_Model"):
    mlflow.log_param("model", "XGBoost")
    mlflow.log_params(best_config)

    mlflow.log_param("encoding", "frequency_encoding")
    mlflow.log_param("feature_selection", "xgboost_importance_top200 + correlation_filter")
    mlflow.log_param("num_final_features", X_train_final.shape[1])

    mlflow.log_metric("best_valid_auc", best_valid_auc)

    mlflow.xgboost.log_model(best_xgb, artifact_path="xgboost_best_model")

print("Best config:", best_config)
print("Best validation AUC:", best_valid_auc)

2026/05/02 18:19:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost_BEST_Model at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3/runs/e7ebe776323d4283909463ae6fb77b0d
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3
Best config: {'name': 'XGB_depth8_lr002', 'n_estimators': 1000, 'max_depth': 8, 'learning_rate': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'gamma': 0.1, 'reg_lambda': 2}
Best validation AUC: 0.9481597170458648


In [110]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

class XGBPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, cols_to_drop, categorical_cols, selected_features):
        self.cols_to_drop = cols_to_drop
        self.categorical_cols = categorical_cols
        self.selected_features = selected_features
        self.freq_maps = {}

    def fit(self, X, y=None):
        X = X.copy()
        X.columns = X.columns.str.replace("-", "_")

        X = X.drop(columns=self.cols_to_drop, errors="ignore")
        X = self._add_features(X)

        for col in self.categorical_cols:
            if col in X.columns:
                self.freq_maps[col] = X[col].value_counts(dropna=False).to_dict()

        return self

    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace("-", "_")

        X = X.drop(columns=self.cols_to_drop, errors="ignore")
        X = self._add_features(X)

        for col in self.categorical_cols:
            if col in X.columns:
                X[col + "_freq"] = X[col].map(self.freq_maps.get(col, {})).fillna(0)

        X = X.drop(columns=self.categorical_cols, errors="ignore")

        for col in self.selected_features:
            if col not in X.columns:
                X[col] = np.nan

        X = X[self.selected_features]

        return X

    def _add_features(self, X):
        X = X.copy()

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])

        if "TransactionDT" in X.columns:
            X["TransactionDT_days"] = X["TransactionDT"] / (3600 * 24)
            X["TransactionDT_hours"] = X["TransactionDT"] / 3600

        X["num_missing"] = X.isnull().sum(axis=1)

        return X

In [111]:
final_pipeline = Pipeline(steps=[
    ("preprocessor", XGBPreprocessor(
        cols_to_drop=cols_to_drop,
        categorical_cols=categorical_cols,
        selected_features=X_train_final.columns.tolist()
    )),
    ("model", XGBClassifier(
        **best_config,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

In [112]:
X_full_raw = train.drop(columns=["isFraud"])
y_full = train["isFraud"]

final_pipeline.fit(X_full_raw, y_full)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:24:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "name" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Pipeline(steps=[('preprocessor',
                 XGBPreprocessor(categorical_cols=['ProductCD', 'card4',
                                                   'card6', 'P_emaildomain',
                                                   'R_emaildomain', 'M1', 'M2',
                                                   'M3', 'M4', 'M5', 'M6', 'M7',
                                                   'M8', 'M9', 'id_12', 'id_15',
                                                   'id_16', 'id_28', 'id_29',
                                                   'id_30', 'id_31', 'id_33',
                                                   'id_34', 'id_35', 'id_36',
                                                   'id_37', 'id_38',
                                                   'DeviceType', 'DeviceInfo'],
                                 cols_to_drop=['dist2', 'D7', 'id_07', 'id_08',
                                               'id_1...
                               feature_types=None, feature_weights=None,
                               gamma=0.1, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.02,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=8, max_leaves=None, min_child_weight=5,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=1000,
                               n_jobs=-1, name='XGB_depth8_lr002', ...))])

In [113]:
with mlflow.start_run(run_name="XGBoost_FINAL_Pipeline"):
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("pipeline", True)
    mlflow.log_param("uses_full_data", True)
    mlflow.log_params(best_config)

    mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path="xgboost_pipeline",
        registered_model_name="XGBoost_Fraud_Model"
    )

2026/05/02 18:28:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 18:28:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoost_Fraud_Model'.
2026/05/02 18:28:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Fraud_Model, version 1
Created version '1' of model 'XGBoost_Fraud_Model'.


🏃 View run XGBoost_FINAL_Pipeline at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3/runs/10ce061df9ba4f568779313e67deb691
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/3
